In [ ]:
from Preprocess import EEGPreprocessor
import os
from tqdm import tqdm  
import numpy as np
import pandas as pd
import shutil

# Configuration
config={
    'root' : '/teamspace/studios/this_studio/Dataset/BrainLat/',
    'subject_path' : '/teamspace/studios/this_studio/phdResearch/data2n/Subjects',
    'label_path' : '/teamspace/studios/this_studio/phdResearch/data2n/Labels',
    ##############

    'epoch_duration' : 5.0,   # second
    'epoch_overlap' : 0.0,    # No overlap
    'resample_freq' : 128,    # HZ point per second
    'l_freq' : 0.5,
    'h_freq' : 45.0,
    'notch_freq' : 50.0,
    'asr_cutoff' : 20,
    'iclabel_threshold' : 0.90,
    'random_state' : 42,
    'use_pyprep' : False,
    'flat_std_thresh' : 1.5e-6,  
    'verbose' : False,
    
    ##############
    
}

if os.path.exists(config['subject_path']):
    shutil.rmtree(config['subject_path'])
    shutil.rmtree(config['label_path'])
if not os.path.exists(config['subject_path']):
    os.makedirs(config['subject_path'])
if not os.path.exists(config['label_path']):
    os.makedirs(config['label_path'])
    
#####################################################################################
bad_subject_list = []
label_list = []
AD_data = []
HC_data =[]

sub_id = 1 

for folderlabel in os.listdir(config['root']):
    successful = 0
    failed = 0
    if "BrainLat" in folderlabel or "SYNAPSE" in folderlabel:
        continue
    print(folderlabel)
    folderlabel_path = os.path.join(config['root'], folderlabel)
    for file_folder in os.listdir(folderlabel_path):
        if file_folder == 'AR' or file_folder == 'CL':
            print(file_folder)
            file_folder_path = os.path.join(folderlabel_path, file_folder)
            for file in os.listdir(file_folder_path):
                file_path = os.path.join(file_folder_path, file)
                set_file_path = ''
                if 'sub' in file_path or 'suj' in file_path:   # different subject folder name
                    set_file_path = os.path.join(file_path, 'eeg')
                    if not os.path.exists(set_file_path):  # same subjects has different folder structure
                        set_file_path = file_path
                    print(set_file_path)
                    ########################################################################################
                    for set_file in os.listdir(set_file_path):
                        if set_file.endswith('.set'):
                            set_file_path = os.path.join(set_file_path, set_file)
                            print(set_file_path)
                            #####################################################################
                            try:
                                # Initialize preprocessor
                                preprocessor = EEGPreprocessor(
                                               epoch_duration = config['epoch_duration'],
                                               epoch_overlap = config['epoch_overlap'],
                                               resample_freq = config['resample_freq'],
                                               l_freq = config['l_freq'],
                                               h_freq = config['h_freq'],
                                               notch_freq = config['notch_freq'],
                                               asr_cutoff = config['asr_cutoff'],
                                               iclabel_threshold = config['iclabel_threshold'],
                                               random_state = config['random_state'],
                                               flat_std_thresh = config['flat_std_thresh'],  
                                               verbose = config['verbose']
                                              )                        
                                if 'reject' in set_file_path:
                                    print('Load EEG data error, skip this subject\n')
                                    
                                else:    
                                    # Process file
                                    if 'HC' in folderlabel or 'AD' in folderlabel:
                                        data = preprocessor.preprocess(set_file_path)
                
                                        print(data.shape)
                                        e,c,t=data.shape
                                       
                                        np.save(config['subject_path'] + '/Sub_{:03d}.npy'.format(sub_id), data)
                                        successful += 1
                                        if 'HC' in folderlabel:
                                            label_list.append(np.array([sub_id,0]))
                                        elif 'AD' in folderlabel:
                                            label_list.append(np.array([sub_id,1]))
                                        
                                        print(f"Raw subject ID: {file}")
                                        print(f"Non-empty subject ID: {sub_id}")
                                        sub_id += 1
                                        print('\n')
                            except:
                                bad_subject_list.append(set_file_path)
                              
                                failed += 1
                                continue       
                        else:  
                            continue
    print("-------------------------------------\n")
    print(f"Processing complete")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    
labels = np.array(label_list)

labels_df = pd.DataFrame(labels, columns=['subject_id', 'label'])
labels_df['subject_id'] = [f'Sub_{i:03d}' for i in labels_df['subject_id']]
condition_mapping = {0: 'HC', 1: 'AD'}
labels_df['Group'] = labels_df['label'].map(condition_mapping)
csv_path = os.path.join(config['label_path'], 'labels.csv')
labels_df.to_csv(csv_path, index=False)
print(f"Labels saved to: {csv_path}")

1_AD
AR
/teamspace/studios/this_studio/Dataset/BrainLat/1_AD/AR/sub-30001/eeg
/teamspace/studios/this_studio/Dataset/BrainLat/1_AD/AR/sub-30001/eeg/s6_sub-30001_rs-HEP_eeg.set

[Pipeline Start] -> /teamspace/studios/this_studio/Dataset/BrainLat/1_AD/AR/sub-30001/eeg/s6_sub-30001_rs-HEP_eeg.set
-> High-density EEG detected
Applying spatial layout harmonization...
Harmonized channels count: 19/19
-> Band-pass filtering
-> Removing line noise
-> Resampling to 128 Hz
-> PyPREP disabled.
-> Selecting optimal ASR calibration window...
[Success] Optimal ASR baseline selected at 25th percentile: 240.00s to 270.00s
(19, 3969)
---------------------
Shape: (19, 3969)
dtype: float64
NaN: 0
Inf: 0
Samples: 3969
---------------------
-> Variance ratio : 0.992
-> eeg_reference average 
-> Fitting ICA...
-> Running ICLabel...

ICLabel Classification
IC 00 | brain              | 0.997
IC 01 | brain              | 1.000
IC 02 | brain              | 0.997
IC 03 | brain              | 0.597
IC 04 | brain 

Traceback (most recent call last):
  File "/teamspace/studios/this_studio/phdResearch/preprocessing/Preprocess.py", line 274, in preprocess
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/eeglab/eeglab.py", line 329, in read_raw_eeglab
    return RawEEGLAB(
  File "<decorator-gen-290>", line 10, in __init__
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/eeglab/eeglab.py", line 467, in __init__
    super().__init__(
  File "<decorator-gen-259>", line 10, in __init__
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/base.py", line 259, in __init__
    self.filenames = list(filenames)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/base.py", line 703, in filenames
    raise FileNotFoundError(f"File {value[k]} not found.")
FileNotFoundError: File /teamspace/studios/this_studio/Dataset/

-> High-density EEG detected
Applying spatial layout harmonization...
Harmonized channels count: 19/19
-> Band-pass filtering
-> Removing line noise
-> Resampling to 128 Hz
-> PyPREP disabled.
-> Selecting optimal ASR calibration window...
[Warning] No perfectly clean window found. Falling back to the beginning.
(19, 3841)
---------------------
Shape: (19, 3841)
dtype: float64
NaN: 0
Inf: 0
Samples: 3841
---------------------
-> Variance ratio : 0.990
-> eeg_reference average 
-> Fitting ICA...
-> Running ICLabel...

ICLabel Classification
IC 00 | brain              | 0.993
IC 01 | brain              | 0.815
IC 02 | brain              | 0.996
IC 03 | brain              | 0.990
IC 04 | brain              | 0.514
IC 05 | brain              | 0.910
IC 06 | brain              | 0.991
IC 07 | brain              | 0.902
IC 08 | brain              | 0.473
IC 09 | other              | 0.902
IC 10 | brain              | 0.830
IC 11 | other              | 0.555
IC 12 | brain              | 0.732

Traceback (most recent call last):
  File "/teamspace/studios/this_studio/phdResearch/preprocessing/Preprocess.py", line 274, in preprocess
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/eeglab/eeglab.py", line 329, in read_raw_eeglab
    return RawEEGLAB(
  File "<decorator-gen-290>", line 10, in __init__
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/eeglab/eeglab.py", line 467, in __init__
    super().__init__(
  File "<decorator-gen-259>", line 10, in __init__
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/base.py", line 259, in __init__
    self.filenames = list(filenames)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/base.py", line 703, in filenames
    raise FileNotFoundError(f"File {value[k]} not found.")
FileNotFoundError: File /teamspace/studios/this_studio/Dataset/

-> High-density EEG detected
Applying spatial layout harmonization...
Harmonized channels count: 19/19
-> Band-pass filtering
-> Removing line noise
-> Resampling to 128 Hz
-> PyPREP disabled.
-> Selecting optimal ASR calibration window...
[Warning] No perfectly clean window found. Falling back to the beginning.
(19, 3841)
---------------------
Shape: (19, 3841)
dtype: float64
NaN: 0
Inf: 0
Samples: 3841
---------------------
-> Variance ratio : 0.970
-> eeg_reference average 
-> Fitting ICA...
-> Running ICLabel...

ICLabel Classification
IC 00 | brain              | 0.998
IC 01 | brain              | 0.999
IC 02 | brain              | 1.000
IC 03 | brain              | 0.998
IC 04 | brain              | 0.985
IC 05 | brain              | 0.947
IC 06 | brain              | 0.631
IC 07 | brain              | 0.465
IC 08 | brain              | 0.927
IC 09 | brain              | 0.799
IC 10 | other              | 0.676
IC 11 | muscle artifact    | 0.729
IC 12 | brain              | 0.980

Traceback (most recent call last):
  File "/teamspace/studios/this_studio/phdResearch/preprocessing/Preprocess.py", line 274, in preprocess
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/eeglab/eeglab.py", line 329, in read_raw_eeglab
    return RawEEGLAB(
  File "<decorator-gen-290>", line 10, in __init__
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/eeglab/eeglab.py", line 467, in __init__
    super().__init__(
  File "<decorator-gen-259>", line 10, in __init__
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/base.py", line 259, in __init__
    self.filenames = list(filenames)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/mne/io/base.py", line 703, in filenames
    raise FileNotFoundError(f"File {value[k]} not found.")
FileNotFoundError: File /teamspace/studios/this_studio/Dataset/

[Error] Pipeline failure: File /teamspace/studios/this_studio/Dataset/BrainLat/5_HC/CL/sub-100025/eeg/s6_sub-100025_rs_eeg.fdt not found.
(0,)
/teamspace/studios/this_studio/Dataset/BrainLat/5_HC/CL/sub-100027/eeg
/teamspace/studios/this_studio/Dataset/BrainLat/5_HC/CL/sub-100027/eeg/s6_sub-100027_rs_eeg.set

[Pipeline Start] -> /teamspace/studios/this_studio/Dataset/BrainLat/5_HC/CL/sub-100027/eeg/s6_sub-100027_rs_eeg.set
[Error] Pipeline failure: File /teamspace/studios/this_studio/Dataset/BrainLat/5_HC/CL/sub-100027/eeg/s6_sub-100027_rs_eeg.fdt not found.
(0,)
/teamspace/studios/this_studio/Dataset/BrainLat/5_HC/CL/sub-100029/eeg
/teamspace/studios/this_studio/Dataset/BrainLat/5_HC/CL/sub-100029/eeg/s6_sub-100029_rs_eeg.set

[Pipeline Start] -> /teamspace/studios/this_studio/Dataset/BrainLat/5_HC/CL/sub-100029/eeg/s6_sub-100029_rs_eeg.set
